# ComplianceGPT — Single Query (S7 Retriever + Live QUR)

This notebook is the **final single-query entrypoint** for ComplianceGPT.

It runs one query end-to-end:

1. **S7 retriever** (variants + hybrid fusion + gated rerank)
      
      1.1 **QUR** (query rewrites; shared model/tokenizer)
3. **Generator** (ID-only selector)
4. **Deterministic fill** (verbatim evidence spans from CCS)
5. Outputs the **Final Answer Contract** (provably extractive)

> Notes  
> - This notebook is **not** an evaluation notebook. It does **not** load gold sets.  
> - Verification against gold labels should be done in a separate notebook.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, sys

# === Project root (Drive) ===
PROJECT_ROOT = "/content/drive/MyDrive/ComplianceGPT"
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("[OK] sys.path configured.")
print("SRC_DIR:", SRC_DIR)

[OK] sys.path configured.
SRC_DIR: /content/drive/MyDrive/ComplianceGPT/src


In [3]:
# === Canonical data paths ===
REV5_CATALOG_PATH = os.path.join(PROJECT_ROOT, "data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl")
REV4_CATALOG_PATH = os.path.join(PROJECT_ROOT, "data/ccs/nist800-53/NIST_SP-800-53_rev4_catalog.jsonl")

ORG_PROFILE_REV5 = os.path.join(PROJECT_ROOT, "data/ODP/rev5/org_profile_example_rev5.yaml")
ORG_PROFILE_REV4 = os.path.join(PROJECT_ROOT, "data/ODP/rev4/org_profile_example_rev4.yaml")

print("[OK] Paths set.")

[OK] Paths set.


In [4]:
# === Run config ===
FRAMEWORK_VERSION = "rev5"         # "rev4" or "rev5"
USE_ORG_PROFILE = False            # True only if test org-profile resolution behaviors

# Model used by BOTH generator and QUR (shared weights in memory)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# How many evidence spans to return (contract-level)
TOP_K = 12

# Optional: reduce noisy HF logs
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

print("[OK] Config ready:", {"framework_version": FRAMEWORK_VERSION, "use_org_profile": USE_ORG_PROFILE})

[OK] Config ready: {'framework_version': 'rev5', 'use_org_profile': False}


In [5]:
# --- Build pipeline (S7 retriever + live QUR) ---
from compliancegpt.pipeline.pipeline import ComplianceGPTPipeline
from compliancegpt.retriever.retriever_s7 import RetrievalConfig

fw = FRAMEWORK_VERSION.strip().lower()
if fw not in {"rev4", "rev5"}:
    raise ValueError(f"FRAMEWORK_VERSION must be 'rev4' or 'rev5' (got {FRAMEWORK_VERSION!r})")

ccs_path = REV5_CATALOG_PATH if fw == "rev5" else REV4_CATALOG_PATH
org_profile = (ORG_PROFILE_REV5 if fw == "rev5" else ORG_PROFILE_REV4) if USE_ORG_PROFILE else None
if org_profile and (not os.path.exists(org_profile)):
    print(f"[WARN] org_profile not found; continuing without profile: {org_profile}")
    org_profile = None

# Minimal, safe defaults.
# NOTE: do not mutate RetrievalConfig in-place (it may be frozen); pass desired values at construction time.
retr_cfg = RetrievalConfig()

pipe = ComplianceGPTPipeline(
    framework_version=fw,
    model_id=MODEL_ID,
    ccs_path=ccs_path,
    use_qur=True,                      # <-- Standard ComplianceGPT single run uses QUR
    doc_filter_mode="prefer_smt_keep_params",
    org_profile_path=org_profile,      # optional
    retriever_config=retr_cfg,
    verify_strict_extras=False,
    verify_strict_verbatim=True,
    verify_strict_version=False,
)

print("[OK] Pipeline initialized.")
print("CCS:", ccs_path)

[QUR] RUN_MODE=lib
[QUR] Library mode loaded. (Batch generation & Tests disabled)
QUR library exported as QUR_LIB
[QUR] RUN_MODE=lib


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[OK] Pipeline initialized.
CCS: /content/drive/MyDrive/ComplianceGPT/data/ccs/nist800-53/NIST_SP-800-53_rev5_catalog.jsonl


## Run one query

In [6]:
# === Query ===
QUERY_TEXT = "When must account managers and the designated personnel be notified about user account changes?"
print("[OK] Query set.")

[OK] Query set.


In [7]:
# --- Execute ---
out = pipe.answer(
    QUERY_TEXT,
    top_k=TOP_K,
    use_generator=True,
    run_verify=False,   # single-run demo is not evaluation
)

# --- summary ---
def _preview(s: str, n: int = 360) -> str:
    s = str(s or "")
    s = s.replace("\n", " ").strip()
    return (s[:n] + " ...") if len(s) > n else s

print("\n=== Final Contract (Summary) ===")
print("status:", out.get("status"))
print("odp_required_list:", out.get("odp_required_list"))
print("evidence_spans_len:", len(out.get("evidence_spans") or []))
print("primary_citation:", out.get("primary_citation"))
print("answer_preview:", _preview(out.get("answer_text", "")))

print("\n=== QUR Rewrites Used ===")
dbg = out.get("debug") or {}
for i, r in enumerate(dbg.get("rewrites") or []):
    print(f"[{i}] {r}")

print("\n=== Retriever Meta (Top) ===")
meta = dbg.get("retriever_meta") or {}
for k in sorted(meta.keys()):
    v = meta[k]
    if isinstance(v, (dict, list)) and len(str(v)) > 200:
        continue
    print(f"{k}: {v}")


=== Final Contract (Summary) ===
status: PARAMS_REQUIRED
odp_required_list: ['ac-02_odp.05', 'ac-02_odp.06', 'ac-02_odp.07', 'ac-02_odp.08']
evidence_spans_len: 3
primary_citation: ac-2_obj.h
answer_preview: AC-02h.01 account managers and {{ insert: param, ac-02_odp.05 }} are notified within {{ insert: param, ac-02_odp.06 }} when accounts are no longer required; AC-02h.02 account managers and {{ insert: param, ac-02_odp.05 }} are notified within {{ insert: param, ac-02_odp.07 }} when users are terminated or transferred; AC-02h.03 account managers and {{ insert:  ...

=== QUR Rewrites Used ===
[0] When do account managers and designated personnel need to be informed about changes to user accounts?
[1] What are the requirements for notifying account managers and designated personnel about user account modifications?
[2] Under what circumstances should account managers and designated personnel be made aware of changes in user accounts?

=== Retriever Meta (Top) ===
base_margin_ratio: 0.0

In [8]:
# Optional: inspect evidence spans (verbatim)
SHOW_EVIDENCE = False

if SHOW_EVIDENCE:
    spans = out.get("evidence_spans") or []
    for i, s in enumerate(spans):
        print(f"\n--- Evidence {i} ---")
        print("source_id:", s.get("source_id"))
        print(_preview(s.get("span_text", ""), n=900))